In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, '..')

# 07 — Comparative Analysis & Hypothesis Testing
Load all saved result tables, run H1–H5 paired t-tests, build master summary, and conduct failure analysis.

**Run `run_all.py` first to generate all result tables.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TABLES_DIR = '../results/tables'
FIGURES_DIR = '../results/figures'

def load_table(name):
    path = os.path.join(TABLES_DIR, f'{name}.csv')
    if os.path.exists(path):
        return pd.read_csv(path)
    print(f'WARNING: {name}.csv not found — run run_all.py first')
    return pd.DataFrame()

print('Available tables:', os.listdir(TABLES_DIR) if os.path.exists(TABLES_DIR) else 'none')

In [ ]:
# Table 1: Pipeline × Dataset
t1 = load_table('table1_pipeline_dataset')
print('=== Table 1: Pipeline × Dataset ===')
print(t1.to_string(index=False))

In [ ]:
# Table 2: Sparsity
t2 = load_table('table2_sparsity')
print('=== Table 2: Sparsity ===')
print(t2.to_string(index=False))

In [ ]:
# Table 3: Regularization
t3 = load_table('table3_regularization')
print('=== Table 3: Regularization (best C per type per dataset, Pipeline A) ===')
if not t3.empty:
    subset = t3[t3['Pipeline'] == 'A']
    best = subset.loc[subset.groupby(['Regularization', 'Dataset'])['pr_auc_mean'].idxmax()]
    print(best[['Regularization', 'Dataset', 'C', 'pr_auc_mean', 'pr_auc_std']].to_string(index=False))

In [ ]:
# Table 4: Imbalance × Regularization
t4 = load_table('table4_imbalance_interaction')
print('=== Table 4: Imbalance × Regularization ===')
print(t4.to_string(index=False))

In [ ]:
# H1: Pipeline effect
h1 = load_table('h1_pipeline_ttest')
print('=== H1: Pipeline Effect (paired t-test) ===')
print(h1.to_string(index=False))
if not h1.empty:
    sig = h1[h1['p_value'] < 0.05]
    print(f'\nSignificant pairs (p<0.05): {len(sig)}/{len(h1)}')

In [ ]:
# H2: Regularization generalization
h2 = load_table('h2_regularization_ttest')
print('=== H2: Regularization Generalization ===')
print(h2.to_string(index=False))

In [ ]:
# H3: ElasticNet vs L1/L2
h3 = load_table('h3_elasticnet_ttest')
print('=== H3: ElasticNet vs L1/L2 ===')
print(h3.to_string(index=False))

In [ ]:
# H4: Imbalance × Regularization interaction
h4 = load_table('h4_imbalance_ttest')
print('=== H4: Imbalance × Regularization Interaction ===')
print(h4.to_string(index=False))

In [ ]:
# H5: Polynomial overfitting
h5 = load_table('h5_polynomial_ttest')
print('=== H5: Polynomial Overfitting ===')
print(h5.to_string(index=False))

In [ ]:
# Holdout final evaluation
holdout_df = load_table('holdout_final_evaluation')
print('=== Final Holdout Evaluation (Pipeline A, L2, C=1) ===')
print(holdout_df.to_string(index=False))

In [ ]:
# Master summary: p-value heatmap across all hypotheses
all_h = {'H1': h1, 'H2': h2, 'H3': h3, 'H4': h4, 'H5': h5}
summary_rows = []
for h_name, df in all_h.items():
    if not df.empty and 'p_value' in df.columns:
        for _, row in df.iterrows():
            summary_rows.append({
                'Hypothesis': h_name,
                'Comparison': f"{row['Group A']} vs {row['Group B']}",
                'p-value': round(row['p_value'], 4),
                'Significant (p<0.05)': '✓' if row['p_value'] < 0.05 else '✗',
            })

if summary_rows:
    master = pd.DataFrame(summary_rows)
    print('=== Master Hypothesis Summary ===')
    print(master.to_string(index=False))

In [ ]:
# Failure analysis: identify worst-performing configurations
if not t3.empty:
    print('=== Failure Analysis: Worst PR-AUC configurations ===')
    worst = t3.nsmallest(5, 'pr_auc_mean')[['Regularization', 'C', 'Dataset', 'Pipeline', 'pr_auc_mean']]
    print(worst.to_string(index=False))

    print('\n=== Best PR-AUC configurations ===')
    best = t3.nlargest(5, 'pr_auc_mean')[['Regularization', 'C', 'Dataset', 'Pipeline', 'pr_auc_mean']]
    print(best.to_string(index=False))

In [ ]:
# Bias-variance narrative summary table
narrative = pd.DataFrame([
    {'Method': 'No Regularization', 'Bias': 'Low', 'Variance': 'High', 'Generalization': 'Poor (overfits)'},
    {'Method': 'L1 (Lasso)', 'Bias': 'Higher', 'Variance': 'Lower', 'Generalization': 'Medium (sparse)'},
    {'Method': 'L2 (Ridge)', 'Bias': 'Moderate', 'Variance': 'Moderate', 'Generalization': 'Good (stable)'},
    {'Method': 'Elastic Net', 'Bias': 'Balanced', 'Variance': 'Balanced', 'Generalization': 'Best (expected)'},
    {'Method': 'PCA preprocessing', 'Bias': 'Slight ↑', 'Variance': '↓ Reduces', 'Generalization': 'Improves'},
    {'Method': 'SMOTE', 'Bias': '↓ on minority', 'Variance': '↑ synthetic noise', 'Generalization': 'Better recall'},
])
print('=== Bias-Variance Narrative Summary ===')
print(narrative.to_string(index=False))